# UniversalXAS activation and seed sweep

This notebook trains nine UniversalXAS heads on fixed exported 64D FEFF features. It does not retrain the encoder.

Validation eta selects one global candidate. Test metrics are computed only for that candidate.

## 1. Source data and sweep settings

In [1]:
from __future__ import annotations

import json
import os
import random
import sys
from pathlib import Path

import lightning.pytorch as pl
import numpy as np
import pandas as pd
import torch
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from torch import nn
from torch.utils.data import DataLoader, TensorDataset


def find_repo_root() -> Path:
    candidates = []
    if os.environ.get("OMNIXAS_REPO_ROOT"):
        candidates.append(Path(os.environ["OMNIXAS_REPO_ROOT"]).expanduser())
    candidates += [Path.cwd(), *Path.cwd().parents]
    candidates += [
        Path.home() / "Desktop" / "OmniXAS",
        Path("/mnt/c/Users/anton/Desktop/OmniXAS"),
        Path("/c/Users/anton/Desktop/OmniXAS"),
    ]
    for candidate in candidates:
        candidate = candidate.resolve()
        if (candidate / "pyproject.toml").is_file() and (candidate / "omnixas").is_dir():
            return candidate
    raise FileNotFoundError("Could not find the OmniXAS repository. Set OMNIXAS_REPO_ROOT.")


REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT))

from omnixas.data.ml_data import MLData, MLSplits
from omnixas.model.training import LightningXASData, PlModule

FEFF_TASKS = ["Ti_FEFF", "V_FEFF", "Cr_FEFF", "Mn_FEFF", "Fe_FEFF", "Co_FEFF", "Ni_FEFF", "Cu_FEFF"]
SPLITS = ["train", "val", "test"]
INPUT_DIM, OUTPUT_DIM = 64, 141
HIDDEN_DIMS = [500, 500, 550]
ACTIVATIONS = ["SiLU", "GELU", "LeakyReLU(0.1)"]
SEEDS = [44, 45, 46]
RESUME_EXISTING = True

ARCHIVE_RUN = Path(
    os.environ.get(
        "OMNIXAS_E2E_UNIVERSAL_RUN",
        REPO_ROOT.parent / "fulltrainingcopy072726" / "m3gnetAll8E2EUniversal" / "e2e_universal_seed42",
    )
).expanduser().resolve()
FEATURES_DIR = ARCHIVE_RUN / "features"
ID_DIR = REPO_ROOT / "tutorial_omnixas" / "material_id_and_site"
CANONICAL_Y_DIR = REPO_ROOT / "tutorial_omnixas" / "ml_data"
OUTPUT_ROOT = ARCHIVE_RUN / "notebook_activation_sweep"

SETTINGS = {
    "hidden_dims": HIDDEN_DIMS,
    "feature_scale": 1000.0,
    "dropout": 0.10,
    "optimizer": "Adam",
    "learning_rate": 5e-4,
    "scheduler": {"name": "ReduceLROnPlateau", "mode": "min", "factor": 0.5, "patience": 8, "min_lr": 1e-6, "frequency": 2},
    "max_epochs": 800,
    "early_stopping_patience": 60,
    "batch_size": 32,
    "check_val_every_n_epoch": 2,
    "monitor": "val_median_mse",
    "activations": ACTIVATIONS,
    "seeds": SEEDS,
}
CANDIDATES = [
    {"name": f"{activation.lower().replace('(0.1)', '_0p1')}_seed{seed}", "activation": activation, "seed": seed}
    for activation in ACTIVATIONS
    for seed in SEEDS
]
if len(CANDIDATES) != 9 or len({c["name"] for c in CANDIDATES}) != 9:
    raise AssertionError("The sweep must contain exactly nine unique candidates.")
if not FEATURES_DIR.is_dir():
    raise FileNotFoundError(f"Missing exported feature directory: {FEATURES_DIR}")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("source:", ARCHIVE_RUN)
print("features:", FEATURES_DIR)
print("output:", OUTPUT_ROOT)

source: /mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42
features: /mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/features
output: /mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_activation_sweep


## 2. Load and validate fixed feature splits

In [2]:
def read_matrix(path: Path) -> np.ndarray:
    if not path.is_file():
        raise FileNotFoundError(f"Missing data file: {path}")
    array = np.loadtxt(path, dtype=np.float32)
    array = np.atleast_2d(array)
    if array.ndim != 2:
        raise ValueError(f"Expected a 2D array in {path}, got {array.shape}")
    return array


def read_ids(path: Path) -> list[tuple[str, int]]:
    if not path.is_file():
        raise FileNotFoundError(f"Missing material/site ID file: {path}")
    rows = []
    for line_number, line in enumerate(path.read_text(encoding="utf-8").splitlines(), 1):
        value = line.strip()
        if not value:
            continue
        try:
            material, site_text = value.rsplit("_", 1)
            site = int(site_text)
        except ValueError as exc:
            raise ValueError(f"Invalid material/site ID at {path}:{line_number}: {value}") from exc
        if not material or site < 0:
            raise ValueError(f"Invalid material/site ID at {path}:{line_number}: {value}")
        rows.append((material, site))
    if len(rows) != len(set(rows)):
        raise ValueError(f"Duplicate material/site IDs in {path}")
    return rows


def load_split(task: str, split_name: str) -> tuple[MLData, list[tuple[str, int]]]:
    X = read_matrix(FEATURES_DIR / f"{task}_{split_name}_X.txt")
    y = read_matrix(FEATURES_DIR / f"{task}_{split_name}_y.txt")
    ids = read_ids(ID_DIR / f"{task}_{split_name}.txt")
    if X.shape != (len(ids), INPUT_DIM) or y.shape != (len(ids), OUTPUT_DIM):
        raise ValueError(f"Invalid {task} {split_name}: IDs={len(ids)}, X={X.shape}, y={y.shape}")
    if not np.isfinite(X).all() or not np.isfinite(y).all():
        raise ValueError(f"Non-finite values in {task} {split_name}")
    canonical_y = read_matrix(CANONICAL_Y_DIR / f"{task}_{split_name}_y.txt")
    if canonical_y.shape != y.shape or not np.allclose(y, canonical_y, rtol=0.0, atol=1e-6):
        raise ValueError(f"Exported targets are not row-aligned with the established IDs for {task} {split_name}")
    return MLData(X=X, y=y), ids


splits: dict[str, MLSplits] = {}
ids_by_task: dict[str, dict[str, list[tuple[str, int]]]] = {}
for task in FEFF_TASKS:
    loaded = {split_name: load_split(task, split_name) for split_name in SPLITS}
    splits[task] = MLSplits(**{name: item[0] for name, item in loaded.items()})
    ids_by_task[task] = {name: item[1] for name, item in loaded.items()}
    materials = {name: {material for material, _ in rows} for name, rows in ids_by_task[task].items()}
    for left, right in (("train", "val"), ("train", "test"), ("val", "test")):
        overlap = materials[left] & materials[right]
        if overlap:
            raise ValueError(f"Material split isolation failed for {task}: {left}/{right} overlap {sorted(overlap)[:3]}")

def combine_tasks(tasks: list[str]) -> MLSplits:
    return MLSplits(**{
        split_name: MLData(
            X=np.concatenate([getattr(splits[task], split_name).X for task in tasks]),
            y=np.concatenate([getattr(splits[task], split_name).y for task in tasks]),
        )
        for split_name in SPLITS
    })

universal_feff = combine_tasks(FEFF_TASKS)
if len(universal_feff.train) != sum(len(splits[task].train) for task in FEFF_TASKS):
    raise AssertionError("Combined UniversalXAS training rows are misaligned.")
print(pd.DataFrame([{"dataset": task, **{split_name: len(getattr(splits[task], split_name)) for split_name in SPLITS}} for task in FEFF_TASKS]).to_string(index=False))
print("combined train/val/test:", [len(getattr(universal_feff, name)) for name in SPLITS])

dataset  train  val  test
Ti_FEFF   5140  641   641
 V_FEFF   8653 1080  1080
Cr_FEFF   2457  305   305
Mn_FEFF  13644 1704  1704
Fe_FEFF   9657 1205  1205
Co_FEFF   8605 1074  1074
Ni_FEFF   3471  432   432
Cu_FEFF   3340  416   416
combined train/val/test: [54967, 6857, 6857]


## 3. Configurable XASBlock and safe candidate training

The project XASBlock always uses SiLU. This local equivalent changes only the hidden activation. The output activation remains Softplus.

In [3]:
def make_activation(name: str) -> nn.Module:
    if name == "SiLU":
        return nn.SiLU()
    if name == "GELU":
        return nn.GELU()
    if name == "LeakyReLU(0.1)":
        return nn.LeakyReLU(0.1)
    raise ValueError(f"Unsupported hidden activation: {name}")


class ConfigurableXASBlock(nn.Sequential):
    def __init__(self, input_dim: int, hidden_dims: list[int], output_dim: int, activation: str, dropout: float):
        if not hidden_dims or not 0.0 <= dropout < 1.0:
            raise ValueError("hidden_dims must be non-empty and dropout must be in [0, 1)")
        dims = [input_dim, *hidden_dims, output_dim]
        layers: list[nn.Module] = []
        for index, (width_in, width_out) in enumerate(zip(dims[:-1], dims[1:])):
            layers.append(nn.Linear(width_in, width_out))
            if index < len(dims) - 2:
                layers.extend([nn.BatchNorm1d(width_out), make_activation(activation), nn.Dropout(dropout)])
            else:
                layers.append(nn.Softplus())
        super().__init__(*layers)


def candidate_checkpoint(directory: Path) -> Path | None:
    best = sorted(directory.glob("best*.ckpt")) if directory.exists() else []
    if len(best) > 1:
        raise RuntimeError(f"Completed candidate must contain exactly one best checkpoint: {directory}")
    if len(best) == 1:
        return best[0]
    if directory.exists() and any(directory.iterdir()):
        raise RuntimeError(f"Incomplete non-empty candidate directory has no best checkpoint: {directory}")
    return None


def write_settings_once(directory: Path, settings: dict) -> None:
    directory.mkdir(parents=True, exist_ok=True)
    path = directory / "settings.json"
    if path.exists():
        if json.loads(path.read_text(encoding="utf-8")) != settings:
            raise ValueError(f"Candidate settings mismatch: {path}")
    else:
        path.write_text(json.dumps(settings, indent=2, sort_keys=True), encoding="utf-8")


def train_candidate(candidate: dict) -> Path:
    directory = OUTPUT_ROOT / "candidates" / candidate["name"]
    metadata = {"stage": "UniversalXAS_activation_seed_sweep", "candidate": candidate, "source_features": str(FEATURES_DIR), "tasks": FEFF_TASKS, "settings": SETTINGS}
    existing = candidate_checkpoint(directory)
    if existing is not None and RESUME_EXISTING:
        write_settings_once(directory, metadata)
        return existing
    if existing is not None:
        raise FileExistsError(f"Candidate is already complete: {directory}")
    write_settings_once(directory, metadata)
    seed = int(candidate["seed"])
    pl.seed_everything(seed, workers=True)
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    model = ConfigurableXASBlock(INPUT_DIM, HIDDEN_DIMS, OUTPUT_DIM, candidate["activation"], SETTINGS["dropout"])
    module = PlModule(
        model,
        lr=SETTINGS["learning_rate"],
        lr_scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau,
        lr_scheduler_kwargs={"mode": "min", "factor": 0.5, "patience": 8, "min_lr": 1e-6},
        lr_scheduler_interval="epoch",
        lr_scheduler_frequency=2,
        lr_scheduler_monitor=SETTINGS["monitor"],
    )
    data = LightningXASData(ml_splits=universal_feff, batch_size=SETTINGS["batch_size"], shuffle=True)
    checkpoint = ModelCheckpoint(
        dirpath=str(directory), filename="best-{epoch:03d}-{val_median_mse:.8f}", monitor=SETTINGS["monitor"], mode="min", save_top_k=1, save_last=True, auto_insert_metric_name=False,
    )
    trainer = pl.Trainer(
        max_epochs=SETTINGS["max_epochs"], accelerator="auto", devices=1, check_val_every_n_epoch=SETTINGS["check_val_every_n_epoch"],
        callbacks=[EarlyStopping(monitor=SETTINGS["monitor"], mode="min", patience=SETTINGS["early_stopping_patience"]), checkpoint],
        default_root_dir=str(directory), logger=False, enable_progress_bar=False, log_every_n_steps=1,
    )
    trainer.fit(module, data)
    result = candidate_checkpoint(directory)
    if result is None:
        raise RuntimeError(f"Training finished without a best checkpoint: {directory}")
    return result

## 4. Validation sweep and global selection

In [4]:
@torch.inference_mode()
def predict_checkpoint(checkpoint: Path, X: np.ndarray, activation: str) -> np.ndarray:
    module = PlModule.load_from_checkpoint(
        checkpoint_path=str(checkpoint),
        model=ConfigurableXASBlock(INPUT_DIM, HIDDEN_DIMS, OUTPUT_DIM, activation, SETTINGS["dropout"]),
        lr=SETTINGS["learning_rate"],
    ).eval()
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    module = module.to(device)
    batches = DataLoader(TensorDataset(torch.tensor(X, dtype=torch.float32)), batch_size=1024, shuffle=False)
    predictions = [module(xb.to(device)).cpu().numpy() for (xb,) in batches]
    if not predictions:
        raise ValueError("Cannot evaluate an empty split")
    prediction = np.concatenate(predictions)
    if not np.isfinite(prediction).all():
        raise ValueError(f"Non-finite prediction from {checkpoint}")
    return prediction


def evaluate(checkpoint: Path, split: MLSplits, split_name: str, activation: str) -> dict[str, float]:
    data = getattr(split, split_name)
    prediction = predict_checkpoint(checkpoint, data.X, activation)
    per_spectrum_mse = np.mean((data.y - prediction) ** 2, axis=1)
    train_mean = split.train.y.mean(axis=0, keepdims=True)
    baseline_mse = np.mean((data.y - train_mean) ** 2, axis=1)
    median_mse = float(np.median(per_spectrum_mse))
    baseline_median_mse = float(np.median(baseline_mse))
    if not np.isfinite(median_mse) or not np.isfinite(baseline_median_mse) or median_mse <= 0.0:
        raise ValueError(f"Invalid evaluation metric for {checkpoint} on {split_name}")
    return {
        f"{split_name}_median_mse": median_mse,
        f"{split_name}_baseline_median_mse": baseline_median_mse,
        f"{split_name}_eta": baseline_median_mse / median_mse,
    }


validation_rows = []
for candidate in CANDIDATES:
    checkpoint = train_candidate(candidate)
    for task in FEFF_TASKS:
        row = {"candidate": candidate["name"], "activation": candidate["activation"], "seed": candidate["seed"], "dataset": task, "checkpoint": str(checkpoint)}
        row.update(evaluate(checkpoint, splits[task], "val", candidate["activation"]))
        validation_rows.append(row)

validation_df = pd.DataFrame(validation_rows)
if len(validation_df) != len(CANDIDATES) * len(FEFF_TASKS):
    raise AssertionError("Validation results do not cover all nine candidates and eight datasets.")
RESULTS_DIR = OUTPUT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
validation_path = RESULTS_DIR / "candidate_validation.csv"
summary_path = RESULTS_DIR / "candidate_summary.csv"
validation_df.to_csv(validation_path, index=False)
summary_df = (
    validation_df.groupby(["candidate", "activation", "seed", "checkpoint"], as_index=False)["val_eta"]
    .mean()
    .rename(columns={"val_eta": "mean_val_eta"})
    .sort_values(["mean_val_eta", "candidate"], ascending=[False, True], ignore_index=True)
)
summary_df.to_csv(summary_path, index=False)
selected = summary_df.iloc[0].to_dict()
selected_checkpoint = Path(selected["checkpoint"])
print(summary_df.to_string(index=False))
print("selected checkpoint:", selected_checkpoint)

Seed set to 44
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
You are using a CUDA device ('NVIDIA GeForce RTX 4070 Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
/home/anton/miniconda3/envs/omnixas/lib/python3.11/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_activation_sweep/candidates/silu_seed44 exists and is not 

           candidate     activation  seed                                                                                                                                                                       checkpoint  mean_val_eta
         gelu_seed44           GELU    44          /mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_activation_sweep/candidates/gelu_seed44/best-577-0.00212989.ckpt     16.834777
         silu_seed46           SiLU    46          /mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_activation_sweep/candidates/silu_seed46/best-431-0.00215984.ckpt     16.579853
         gelu_seed45           GELU    45          /mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_activation_sweep/candidates/gelu_seed45/best-385-0.00216076.ckpt     16.465573
         gelu_seed46           GELU    46          /mnt/c/Users/anto

## 5. Selected test evaluation and manifest

In [5]:
test_path = RESULTS_DIR / "selected_test.csv"
if test_path.exists():
    selected_test_df = pd.read_csv(test_path)
    if set(selected_test_df["checkpoint"].astype(str)) != {str(selected_checkpoint)} or set(selected_test_df["dataset"]) != set(FEFF_TASKS):
        raise ValueError(f"Existing selected test artifact does not match the selected checkpoint: {test_path}")
else:
    selected_test_rows = []
    for task in FEFF_TASKS:
        row = {"candidate": selected["candidate"], "activation": selected["activation"], "seed": int(selected["seed"]), "dataset": task, "checkpoint": str(selected_checkpoint)}
        row.update(evaluate(selected_checkpoint, splits[task], "test", selected["activation"]))
        selected_test_rows.append(row)
    selected_test_df = pd.DataFrame(selected_test_rows)
    selected_test_df.to_csv(test_path, index=False)

manifest = {
    "source": {
        "archive_run": str(ARCHIVE_RUN),
        "features_dir": str(FEATURES_DIR),
        "feature_dim": INPUT_DIM,
        "target_dim": OUTPUT_DIM,
        "encoder_retrained": False,
        "id_dir": str(ID_DIR),
    },
    "settings": SETTINGS,
    "tasks": FEFF_TASKS,
    "candidate_count": len(CANDIDATES),
    "selected": {
        "candidate": selected["candidate"],
        "activation": selected["activation"],
        "seed": int(selected["seed"]),
        "checkpoint": str(selected_checkpoint),
        "mean_val_eta": float(selected["mean_val_eta"]),
    },
    "artifacts": {
        "candidate_validation": str(validation_path),
        "candidate_summary": str(summary_path),
        "selected_test": str(test_path),
    },
}
manifest_path = OUTPUT_ROOT / "manifest.json"
if manifest_path.exists():
    if json.loads(manifest_path.read_text(encoding="utf-8")) != manifest:
        raise ValueError(f"Existing manifest does not match this sweep: {manifest_path}")
else:
    manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding="utf-8")
print("validation CSV:", validation_path)
print("summary CSV:", summary_path)
print("selected test CSV:", test_path)
print("manifest:", manifest_path)

validation CSV: /mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_activation_sweep/results/candidate_validation.csv
summary CSV: /mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_activation_sweep/results/candidate_summary.csv
selected test CSV: /mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_activation_sweep/results/selected_test.csv
manifest: /mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/notebook_activation_sweep/manifest.json
